# Tours Food Delivery Market Analysis

This notebook analyzes prospects whose commune name is exactly **Tours**, using the cleaned France-wide lead export. It summarizes venue mix, cuisine and dish tags, contact details, record freshness, and mapped locations to help plan an initial delivery-app launch.

**Official business-data source:** [BANCO — Base Nationale des Commerces Ouverte](https://www.data.gouv.fr/datasets/base-nationale-des-commerces-ouverte). The source records and licensing metadata are stored in `france_banco_raw_data/data.csv`, `france_banco_raw_data/metadata.csv`, and `france_banco_raw_data/license.txt`.

**OpenStreetMap enrichment:** Food-venue fields are taken from the France extract provided by [Geofabrik](https://download.geofabrik.de/europe/france.html). The France-wide workflow cleans the OSM data and matches venues to BANCO by OSM ID. Its export, `france_food_delivery_prospects.csv`, keeps BANCO as the base; matched records can include `cuisine_standard` and `dish` from Geofabrik. Geofabrik-only venues without a BANCO match are not included here.

**Kaggle input:** Kaggle notebooks have separate working directories. Run and save a version of the France-wide notebook, then attach its notebook output to this notebook with **Add Input → Notebook Output Files**. The loader below searches attached inputs for the export.

Cuisine and dish tags are incomplete. Mapped businesses do not confirm that a venue offers delivery, is currently operating, or is interested in joining an app. Verify prospects before outreach or operational decisions.

In [ ]:
# Load the cleaned leads and select Tours by an exact, whitespace-trimmed city-name match.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

source_filename = "france_food_delivery_prospects.csv"
source_path = Path.cwd() / source_filename
if not source_path.is_file():
    kaggle_input_root = Path("/kaggle/input")
    input_matches = sorted(kaggle_input_root.rglob(source_filename)) if kaggle_input_root.is_dir() else []
    if len(input_matches) == 1:
        source_path = input_matches[0]
    elif len(input_matches) > 1:
        raise FileNotFoundError(
            f"Found multiple {source_filename} files under {kaggle_input_root}: {input_matches}. "
            "Keep one France notebook output attached, or set source_path explicitly."
        )
    else:
        raise FileNotFoundError(
            f"Could not find {source_filename} in {Path.cwd()} or under {kaggle_input_root}. "
            "Run and save a version of france_food_delivery_prospects.ipynb, then attach its "
            "Notebook Output Files to this Tours notebook with Add Input."
        )

leads = pd.read_csv(source_path, low_memory=False)
tours = leads[leads["com_nom"].astype("string").str.strip().str.casefold().eq("tours")].copy()
if tours.empty:
    raise ValueError("No rows matched com_nom == 'Tours'. Check the source export and city spelling.")

print(f"Loaded {len(leads):,} France-wide prospects; selected {len(tours):,} for Tours.")
# Cuisine and dish fields come from the cleaned Geofabrik enrichment in the source export.

In [ ]:
# Summarize lead volumes and completeness for the Tours subset.
contact_columns = ["phone", "email", "website"]
contact_values = tours[contact_columns].astype("string").apply(
    lambda column: column.str.strip().replace("", pd.NA)
)
has_any_contact = contact_values.notna().any(axis=1)
has_name = tours["name"].astype("string").str.strip().replace("", pd.NA).notna()

overview = pd.Series({
    "Tours prospects": len(tours),
    "With a name": int(has_name.sum()),
    "With phone, email, or website": int(has_any_contact.sum()),
    "With name and at least one contact method": int((has_name & has_any_contact).sum()),
    "With an address": int(tours["address"].notna().sum()),
    "With coordinates": int(tours[["X", "Y"]].notna().all(axis=1).sum()),
})
display(overview.to_frame("count"))

In [ ]:
# Compare venue categories by count and share of mapped Tours prospects.
venue_mix = tours["type"].value_counts(dropna=False).rename_axis("venue_type").to_frame("prospects")
venue_mix["share_%"] = (venue_mix["prospects"] / len(tours) * 100).round(1)
display(venue_mix)

ax = venue_mix["prospects"].sort_values().plot(kind="barh", figsize=(9, 5), color="teal")
ax.set(title="Mapped food and drink venues in Tours", xlabel="Number of prospects", ylabel="Venue type")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## Cuisine and dish profile in Tours

The lead export already includes cleaned Geofabrik `cuisine_standard` and `dish` fields where a BANCO venue matched. Split only on semicolons, count each venue once per token, and display cuisines and dishes separately. Coverage is based on all Tours prospects; missing tags do not mean a venue has no cuisine.


In [ ]:
import unicodedata

required_columns = {"osm_id", "cuisine_standard", "dish"}
if not required_columns.issubset(tours.columns):
    raise ValueError(f"The France-wide lead export must include {sorted(required_columns)}")

def split_tours_tokens(column):
    token_rows = tours[["osm_id", column]].copy()
    token_rows["label"] = token_rows[column].astype("string").str.split(";")
    token_rows = token_rows.explode("label", ignore_index=True)
    token_rows["label"] = token_rows["label"].map(
        lambda value: unicodedata.normalize("NFKC", value).strip().casefold() if pd.notna(value) else pd.NA
    )
    return token_rows.loc[
        token_rows["label"].notna() & token_rows["label"].ne(""),
        ["osm_id", "label"],
    ].drop_duplicates(["osm_id", "label"])

tours_cuisine_tokens = split_tours_tokens("cuisine_standard")
tours_dish_tokens = split_tours_tokens("dish")
tours_venue_count = tours["osm_id"].nunique()
tours_cuisine_tagged_count = tours_cuisine_tokens["osm_id"].nunique()
tours_cuisine_coverage_pct = tours_cuisine_tagged_count / tours_venue_count * 100 if tours_venue_count else 0
tours_cuisine_counts = tours_cuisine_tokens.groupby("label")["osm_id"].nunique().sort_values(ascending=False)
tours_dish_counts = tours_dish_tokens.groupby("label")["osm_id"].nunique().sort_values(ascending=False)

print(
    f"Tours cuisine-tag coverage: {tours_cuisine_tagged_count:,} / {tours_venue_count:,} venues "
    f"({tours_cuisine_coverage_pct:.1f}%); {tours_cuisine_counts.size:,} cuisine labels."
)
print(f"Tours dish tags: {tours_dish_tokens['osm_id'].nunique():,} venues; dishes are shown separately from cuisines.")

top_tours_cuisines = tours_cuisine_counts.head(12).sort_values()
top_tours_dishes = tours_dish_counts.head(10).sort_values()
fig, (cuisine_ax, dish_ax) = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
top_tours_cuisines.plot(kind="barh", ax=cuisine_ax, color="#367c8b")
cuisine_ax.set(
    title="Standardized cuisines in Tours",
    xlabel="Distinct venues (non-exclusive counts)",
    ylabel="Cuisine",
)
cuisine_ax.grid(axis="x", alpha=0.25)

top_tours_dishes.plot(kind="barh", ax=dish_ax, color="#d08c60")
dish_ax.set(
    title="Top mapped dishes (separate from cuisine)",
    xlabel="Distinct venues (non-exclusive counts)",
    ylabel="",
)
dish_ax.grid(axis="x", alpha=0.25)
fig.suptitle("Tours cuisine and dish profile", fontsize=15, fontweight="bold")
plt.show()

In [ ]:
# Measure each contact channel and compare contactable leads by venue type.
contact_summary = pd.DataFrame({
    "with_field": contact_values.notna().sum(),
    "share_%": (contact_values.notna().mean() * 100).round(1),
})
display(contact_summary)

contact_by_type = tours.assign(has_any_contact=has_any_contact).groupby("type").agg(
    prospects=("type", "size"),
    contactable=("has_any_contact", "sum"),
)
contact_by_type["contactable_%"] = (
    contact_by_type["contactable"] / contact_by_type["prospects"] * 100
).round(1)
display(contact_by_type.sort_values("contactable", ascending=False))

ax = contact_by_type.sort_values("contactable_%")["contactable_%"].plot(
    kind="barh", figsize=(9, 5), color="darkorange"
)
ax.set(title="Contact coverage by venue type in Tours", xlabel="With phone, email, or website (%)", ylabel="Venue type")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Create transparent outreach-review tiers; these indicate data completeness, not delivery suitability.
tours["last_update"] = pd.to_datetime(tours["last_update"], errors="coerce")
today = pd.Timestamp.today().normalize()
tours["update_age_days"] = (today - tours["last_update"]).dt.days
tours["has_name"] = has_name
tours["has_contact"] = has_any_contact

recent = tours["update_age_days"].between(0, 365)
known_name = tours["has_name"]
known_contact = tours["has_contact"]
tours["review_tier"] = "Verify details before outreach"
tours.loc[known_name & known_contact & ~recent, "review_tier"] = "Name and contact; verify freshness"
tours.loc[known_name & known_contact & recent, "review_tier"] = "First-pass outreach review"

tier_summary = tours["review_tier"].value_counts().rename_axis("review_tier").to_frame("prospects")
tier_summary["share_%"] = (tier_summary["prospects"] / len(tours) * 100).round(1)
display(tier_summary)

priority_leads = tours.loc[
    tours["review_tier"] == "First-pass outreach review",
    ["name", "brand", "type", "address", "phone", "email", "website", "opening_hours", "last_update", "osm_id"],
].sort_values(["type", "name"])
priority_lead_preview = priority_leads.head(25).copy()  # Keep the review sample available without rendering raw records.

In [ ]:
# Show how recently Tours prospects were last updated in the source map.
freshness = pd.Series("Unknown date", index=tours.index, dtype="object")
freshness.loc[tours["update_age_days"] < 0] = "Future-dated"
freshness.loc[tours["update_age_days"].between(0, 365)] = "Within 1 year"
freshness.loc[tours["update_age_days"].between(366, 1095)] = "1-3 years"
freshness.loc[tours["update_age_days"] > 1095] = "Over 3 years"
freshness_order = ["Within 1 year", "1-3 years", "Over 3 years", "Unknown date", "Future-dated"]
freshness_counts = freshness.value_counts().reindex(freshness_order, fill_value=0)
display(freshness_counts.rename("prospects").to_frame())

ax = freshness_counts[freshness_counts > 0].plot(kind="bar", figsize=(9, 5), color="slateblue")
ax.set(title="Data freshness for Tours prospects", xlabel="Age of last mapped update", ylabel="Number of prospects")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Venue locations and density

The left panel plots individual venues; the right groups the same coordinates into hexagons to show clusters. These are complementary views of one location analysis.


In [ ]:
# Plot venue coordinates and a hexbin density view to reveal candidate clusters within Tours.
tours["X"] = pd.to_numeric(tours["X"], errors="coerce")
tours["Y"] = pd.to_numeric(tours["Y"], errors="coerce")
mapped_tours = tours.dropna(subset=["X", "Y"])
print(f"Mapped coordinates: {len(mapped_tours):,} of {len(tours):,} Tours prospects.")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(mapped_tours["X"], mapped_tours["Y"], s=8, alpha=0.45, color="royalblue")
axes[0].set(title="Mapped venue locations in Tours", xlabel="Longitude", ylabel="Latitude")
axes[0].set_aspect("equal", adjustable="datalim")
axes[0].grid(alpha=0.2)

density = axes[1].hexbin(mapped_tours["X"], mapped_tours["Y"], gridsize=35, mincnt=1, cmap="magma")
axes[1].set(title="Venue density by coordinate grid", xlabel="Longitude", ylabel="Latitude")
axes[1].set_aspect("equal", adjustable="datalim")
axes[1].grid(alpha=0.2)
fig.colorbar(density, ax=axes[1], label="Mapped venues per hexagon")
fig.tight_layout()
plt.show()

In [ ]:
# Inspect repeated address strings as a rough way to identify shared commercial locations.
address_values = tours["address"].astype("string").str.strip().replace("", pd.NA)
address_coverage = pd.Series({
    "with a non-empty address": int(address_values.notna().sum()),
    "distinct non-empty address strings": int(address_values.dropna().nunique()),
    "with coordinates": int(tours[["X", "Y"]].notna().all(axis=1).sum()),
})
display(address_coverage.to_frame("count"))

shared_addresses = address_values.value_counts().loc[lambda counts: counts > 1].head(20)
display(shared_addresses.rename("mapped_venues").to_frame())

## Suggested Tours launch moves

1. **Start with map clusters, not city-wide assumptions.** Use the coordinate density view to identify compact areas with many mapped venues, then inspect those areas on foot. Coordinates show venue concentration, not customer demand or delivery travel times.
2. **Build a verification shortlist.** Begin with named prospects that have a phone, email, or website and a recent map update. Confirm they are open, independent or authorized to join, and willing to offer delivery before treating them as launch partners.
3. **Prioritize delivery fit through direct research.** Ask about takeaway/delivery experience, menu packaging, prep time, peak hours, average order value, and capacity. The `type` field alone cannot establish delivery suitability; bakeries, bars, and cafes may need a different offer or service window than restaurants and fast food.
4. **Validate shared addresses and chains.** Repeated address strings may indicate food halls, shopping centers, or data duplication. Compare names, coordinates, and SIRETs manually; multiple venues can legitimately share one registered company or address.
5. **Pilot compact delivery zones.** Use verified partner locations and actual courier travel times to set a small initial service area. Track delivery time, cancellations, order frequency, and partner retention before expanding into adjacent districts.
6. **Measure gaps before scaling.** Compare prospect counts with field-verified partner sign-ups and customer demand by zone. This dataset is a mapped-business inventory, not a complete census or demand forecast.

**Geographic caveat:** `com_nom == "Tours"` selects the commune label in the source; it does not include neighboring communes in the Tours metropolitan area. Coordinates use longitude (`X`) and latitude (`Y`).